# HW5: Evaluate Before You Automate

**COMPSS 211 | Fall 2026 | Student copy**

Run a small Gemini classification task, compare it with the HW4 traditional pipeline on the same data, and audit generated code.

**Date/deadline:** Sunday, November 22, 2026 at 11:59 p.m.

Complete every required function and written response. Keep the named outputs so the notebook checks can find them.

## Scenario

The fictional Berkeley Methods Studio is considering an LLM-assisted system for routing campus comments. Before anyone uses it, define what a valid result looks like, make one limited live request, inspect errors, compare methods on the same records, and calculate the security and cost exposure.

## What you will practice

- Request structured output from Gemini with a versioned prompt.
- Compare LLM and traditional predictions on identical documents.
- Manually classify disagreements and failure modes.
- Test generated code against an explicit contract.
- Estimate token, cost, privacy, and maintenance exposure.

## Keep handy

- **Model:** Use `gemini-3.6-flash` unless bCourses announces a replacement.
- **Key:** Read `GEMINI_API_KEY` from the environment. Never print or commit it.
- **Comparison:** Hold documents, labels, and scoring rules constant across methods.
- **Verification:** A generated function is accepted only when tests fail on the flawed version and pass on the repair.

## AI and collaboration policy

Gemini is required only for the specified live step in this assignment. Use your own `GEMINI_API_KEY`, never place it in a notebook, and disclose every AI-assisted step.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## 1. Validate one structured prediction

Implement `validate_prediction(record, expected_id, allowed_labels)`.

A valid record looks like:
`{"document_id": "CAMPUS-001", "label": "transit",
"rationale": "The comment concerns bus service."}`.
A missing field, mismatched ID, disallowed label, or blank rationale
must raise a clear exception.

In [ ]:
ALLOWED_LABELS = {"transit", "study_space", "accessibility", "food", "safety", "services"}
GEMINI_MODEL = "gemini-3.6-flash"
PROMPT_VERSION = "bms-campus-routing-v1"
PROMPT_TEMPLATE = """Classify one synthetic campus comment.
Return JSON with keys document_id, label, and rationale.
label must be one of: accessibility, food, safety, services, study_space, transit.
Comment ID: {document_id}
Comment: {text}"""

def make_gemini_client():
    key = os.getenv("GEMINI_API_KEY")
    if not key:
        raise RuntimeError("Set GEMINI_API_KEY in the environment; never paste it here.")
    from google import genai
    return genai.Client(api_key=key)

print("Credential loading is defined; the key is never stored here.")

In [ ]:
def validate_prediction(record, expected_id, allowed_labels):
    """Validate and return one prediction dictionary.

    Require document_id, label, and a non-empty rationale. Confirm the
    document ID matches expected_id and the label is allowed. Raise a
    clear exception when the contract is violated.
    """
    raise NotImplementedError("Implement validate_prediction")

In [ ]:
# Validate the supplied toy record and display toy_prediction.

## 2. Classify six comments

Implement `classify_comments(sample, client, model, prompt_template)`.

**Required live output:** `live_results`, containing six rows with
`document_id`, `label`, and `rationale`. Set
`COMPSS211_LIVE_API=1` and use your own environment variable. The
offline fixture below keeps ordinary notebook validation deterministic,
but it does not satisfy the live-call requirement.

In [ ]:
def classify_comments(sample, client, model, prompt_template):
    """Return one validated prediction row per input row.

    Format the prompt from each record, request JSON, parse response.text,
    call validate_prediction, and return a DataFrame.
    """
    raise NotImplementedError("Implement classify_comments")

In [ ]:
comments = pd.read_csv(course_data_path("hw4_synthetic_campus_comments.csv"))
fixture = pd.read_csv(course_data_path("hw5_recorded_evaluation_fixture.csv"))
live_sample = comments[["document_id", "text"]].head(6).copy()

In [ ]:
# When COMPSS211_LIVE_API=1, create live_results by calling
# classify_comments on live_sample. During offline work, create a
# clearly labeled six-row fixture preview instead.

## 3. Compare identical documents

Implement `evaluate_predictions(frame, prediction_col)`.

**Required outputs:** `evaluation_summary`, `confusion_table`, and
`disagreements`. Use the same 96 document IDs and human labels for
both the recorded fixture and traditional method.

In [ ]:
evaluation = comments.merge(
    fixture,
    on="document_id",
    validate="one_to_one",
)

In [ ]:
def evaluate_predictions(frame, prediction_col):
    """Return (accuracy, confusion, disagreements).

    Compare prediction_col with human_label for accuracy and the confusion
    table. Disagreements are rows where prediction_col differs from
    traditional_prediction.
    """
    raise NotImplementedError("Implement evaluate_predictions")

In [ ]:
# Create evaluation_summary, confusion_table, and disagreements,
# then display all three outputs.

## 4. Repair and test the review queue

The provided function drops rows and mutates its input. Implement
`mark_for_review(frame)` and write `contract_tests(function)` with
exactly three assertions.

**Required outputs:** a two-row `contract_results` table where the
flawed function fails and the repaired function passes, plus
`review_queue`, containing every row marked for human review.

In [ ]:
def flawed_mark_for_review(frame):
    frame["needs_review"] = False
    return frame.dropna()

contract_sample = evaluation.head(12).copy()
contract_sample.loc[contract_sample.index[0], "text"] = None
expected_review = (
    ~contract_sample["recorded_llm_label"].isin(ALLOWED_LABELS)
    | (
        contract_sample["recorded_llm_label"]
        != contract_sample["traditional_prediction"]
    )
).astype(bool)

def run_contract_tests(function):
    try:
        contract_tests(function)
    except Exception as error:
        return {
            "function": function.__name__,
            "status": "fails",
            "detail": f"{type(error).__name__}: {error}",
        }
    return {
        "function": function.__name__,
        "status": "passes",
        "detail": "all three assertions passed",
    }

In [ ]:
def mark_for_review(frame):
    """Return a copy with boolean needs_review.

    Mark rows when the recorded LLM label is invalid or differs from the
    traditional prediction. Preserve every row and do not mutate frame.
    """
    raise NotImplementedError("Implement mark_for_review")

In [ ]:
def contract_tests(function):
    """Write exactly three assert statements for the contract.

    Test row preservation, non-mutation of the input DataFrame, and the
    required boolean needs_review behavior.
    """
    raise NotImplementedError("Implement three contract tests")

In [ ]:
# Run contract_tests against both functions, create contract_results,
# build review_queue, and display both outputs.

## 5. Explain errors, cost, and deployment risk

Write five to seven sentences. Identify at least two concrete
disagreements, distinguish the synthetic fixture from your live call,
and discuss credentials, token cost, human review, technical debt, and
epistemic debt.

In [ ]:
estimated_tokens = len(comments) * 180
estimated_cost_usd = estimated_tokens / 1_000_000 * 0.50
usage_summary = {
    "documents": len(comments),
    "assumed_tokens_per_document": 180,
    "estimated_tokens": estimated_tokens,
    "illustrative_cost_usd": round(estimated_cost_usd, 4),
}
usage_summary

### Your response

**Prompt:** What evidence would make you pause or stop this deployment?

> Write your response here, then delete this instruction.